# Contare i parametri a mente

*Anatomia di un LLM* · capitolo 5, L'attenzione e il blocco · script `03-11`, `scripts/03_11_conteggio_parametri.py`

C'e' una formula che stima i parametri di un Transformer senza aprire il
codice, e sbaglia di pochissimo. Vale la pena impararla, e vale ancora di piu'
sapere quando smette di valere.

Si esegue con `python3 scripts/03_11_conteggio_parametri.py`
e dura circa 15 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/03_11_conteggio_parametri.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
from llm_anatomy.config import AgoGPTConfig
from llm_anatomy.model import AgoGPT
from llm_anatomy.utils import numero, prepara, tabella, titolo, verifica

prepara()

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


device(type='cpu')

In [3]:
titolo("Da dove viene il dodici")
print("  attenzione: Wq, Wk, Wv, Wo, quattro matrici d x d      ->  4 d^2")
print("  FFN classica: salita d x 4d e discesa 4d x d           ->  8 d^2")
print("  totale per blocco                                      -> 12 d^2")
print("  moltiplicato per N blocchi                             -> 12 N d^2")
print("  restano fuori: embedding, norme, e la testa se non condivisa")


Da dove viene il dodici
-----------------------
  attenzione: Wq, Wk, Wv, Wo, quattro matrici d x d      ->  4 d^2
  FFN classica: salita d x 4d e discesa 4d x d           ->  8 d^2
  totale per blocco                                      -> 12 d^2
  moltiplicato per N blocchi                             -> 12 N d^2
  restano fuori: embedding, norme, e la testa se non condivisa


In [4]:
titolo("Cinque modelli, cinque conti")
righe = []
for n_layer, d_model, n_head in ((2, 128, 4), (4, 192, 6), (6, 384, 6),
                                 (12, 768, 12), (24, 1024, 16)):
    cfg = AgoGPTConfig(vocab_size=93, n_layer=n_layer, d_model=d_model,
                        n_head=n_head, act="gelu", bias=False)
    m = AgoGPT(cfg)
    misurati = m.numero_parametri(senza_embedding=True)
    formula = 12 * n_layer * d_model ** 2
    righe.append([
        f"L{n_layer} d{d_model}",
        numero(misurati),
        numero(formula),
        f"{100 * (misurati - formula) / formula:+.2f}%".replace(".", ","),
    ])
tabella(["modello", "misurati (senza emb.)", "12 N d^2", "scarto"], righe)
print("  lo scarto e' quello delle sole norme: due vettori di d elementi per")
print("  blocco piu' uno finale, cioe' un termine lineare in d che scompare")
print("  appena il modello cresce")


Cinque modelli, cinque conti
----------------------------


modello    misurati (senza emb.)     12 N d^2  scarto
---------  ---------------------  -----------  ------
L2 d128                  393.856      393.216  +0,16%
L4 d192                1.771.200    1.769.472  +0,10%
L6 d384               10.621.824   10.616.832  +0,05%
L12 d768              84.953.856   84.934.656  +0,02%
L24 d1024            302.040.064  301.989.888  +0,02%
  lo scarto e' quello delle sole norme: due vettori di d elementi per
  blocco piu' uno finale, cioe' un termine lineare in d che scompare
  appena il modello cresce


In [5]:
titolo("Verifica")
cfg = AgoGPTConfig(vocab_size=93, n_layer=12, d_model=768, n_head=12,
                    act="gelu", bias=False)
m = AgoGPT(cfg)
misurati = m.numero_parametri(senza_embedding=True)
formula = 12 * 12 * 768 ** 2
verifica(abs(misurati - formula) / formula < 0.01,
         "la formula 12 N d^2 e' esatta entro l'1% sulla configurazione da 12 blocchi")


Verifica
--------


  [verifica superata] la formula 12 N d^2 e' esatta entro l'1% sulla configurazione da 12 blocchi


In [6]:
titolo("Dove smette di valere")
righe = []
for nome, kwargs in (
    ("classica (riferimento)", dict(act="gelu")),
    ("SwiGLU", dict(act="swiglu")),
    ("GQA a 4 gruppi", dict(act="gelu", n_kv_head=4)),
    ("SwiGLU + GQA", dict(act="swiglu", n_kv_head=4)),
):
    cfg = AgoGPTConfig(vocab_size=93, n_layer=12, d_model=768, n_head=12,
                        bias=False, **kwargs)
    m = AgoGPT(cfg)
    misurati = m.numero_parametri(senza_embedding=True)
    righe.append([nome, cfg.d_ff, numero(misurati),
                  f"{misurati / formula:.3f}".replace(".", ",")])
tabella(["variante", "d_ff", "parametri", "rispetto a 12 N d^2"], righe)
print("  SwiGLU resta vicinissimo, perche' 8/3 e' scelto apposta per questo;")
print("  GQA toglie parametri dall'attenzione e porta il rapporto sotto uno")


Dove smette di valere
---------------------


variante                d_ff   parametri  rispetto a 12 N d^2
----------------------  ----  ----------  -------------------
classica (riferimento)  3072  84.953.856                1,000
SwiGLU                  2048  84.953.856                1,000
GQA a 4 gruppi          3072  75.516.672                0,889
SwiGLU + GQA            2048  75.516.672                0,889
  SwiGLU resta vicinissimo, perche' 8/3 e' scelto apposta per questo;
  GQA toglie parametri dall'attenzione e porta il rapporto sotto uno


In [7]:
titolo("Quello che la formula ignora")
righe = []
for vocab in (93, 8000, 32000, 128000):
    for n_layer, d_model in ((4, 192), (12, 768)):
        cfg = AgoGPTConfig(vocab_size=vocab, n_layer=n_layer, d_model=d_model,
                            n_head=max(2, d_model // 64), act="gelu", bias=False)
        m = AgoGPT(cfg)
        emb = cfg.vocab_size * cfg.d_model
        tot = m.numero_parametri()
        righe.append([numero(vocab), f"L{n_layer} d{d_model}", numero(tot),
                      f"{100 * emb / tot:.1f}%".replace(".", ",")])
tabella(["vocabolario", "modello", "parametri totali", "quota dell'embedding"],
        righe)
print("  con un vocabolario reale e un modello piccolo l'embedding e' meta' del")
print("  totale: e' il motivo per cui il conteggio 'senza embedding' e' quello")
print("  che compare nelle leggi di scala")


Quello che la formula ignora
----------------------------


vocabolario   modello  parametri totali  quota dell'embedding
-----------  --------  ----------------  --------------------
93            L4 d192         1.789.056                  1,0%
93           L12 d768        85.025.280                  0,1%
8.000         L4 d192         3.307.200                 46,4%
8.000        L12 d768        91.097.856                  6,7%
32.000        L4 d192         7.915.200                 77,6%
32.000       L12 d768       109.529.856                 22,4%
128.000       L4 d192        26.347.200                 93,3%
128.000      L12 d768       183.257.856                 53,6%
  con un vocabolario reale e un modello piccolo l'embedding e' meta' del
  totale: e' il motivo per cui il conteggio 'senza embedding' e' quello
  che compare nelle leggi di scala


## Da provare

aggiungi una configurazione con vocabolario da 128.000 simboli.